Importacion de la librería pandas

In [ ]:
import pandas as pd
print("Pandas funcionando correctamente")

Pandas funcionando correctamente


Importacion del dataset

In [ ]:
df = pd.read_csv(
    "Dataset_Salud_BIGDATA.csv",
    sep=";"
)

print("Registros:", len(df))
print("Columnas:", len(df.columns))

df.head()

Registros: 4791
Columnas: 17


/usr/local/lib/python3.13/dist-packages/google/colab/_dataframe_summarizer.py:88: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  cast_date_col = pd.to_datetime(column, errors="coerce")
/usr/local/lib/python3.13/dist-packages/google/colab/_dataframe_summarizer.py:88: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  cast_date_col = pd.to_datetime(column, errors="coerce")


,cod_pc,fecha_nacimiento,Id_Financiador,valor_labo,fecha_dosaje_1,hemoglobina_1,fecha_dosaje_2,hemoglobina_2,fecha_dosaje_3,hemoglobina_3,fecha_dosaje_4,hemoglobina_4,fecha_dosaje_5,hemoglobina_5,fecha_supT1,fecha_supT2,fecha_supT3
0,*****728,1/08/1998,2,LEV,30/12/2024,12.5,29/01/2025,12.6,NaN,NaN,NaN,NaN,NaN,NaN,30/12/2024,29/01/2025,28/02/2025
1,*****636,15/08/1999,2,MOD,27/03/2024,9.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27/03/2024,11/04/2024,NaN
2,*****786,27/02/2002,2,LEV,14/02/2025,12.8,17/03/2025,10.6,21/04/2025,13.5,NaN,NaN,NaN,NaN,14/02/2025,17/03/2025,NaN
3,*****917,19/12/1989,2,LEV,8/03/2025,12.3,7/04/2025,12.3,8/05/2025,12.4,NaN,NaN,NaN,NaN,8/03/2025,7/04/2025,8/05/2025
4,*****096,25/08/1987,2,LEV,17/11/2023,12.8,18/12/2023,12.8,16/01/2024,12.8,21/02/2024,13.9,NaN,NaN,17/11/2023,18/12/2023,16/01/2024


Copia del dataset en el cual se trabajará

In [ ]:
df_etl = df.copy()
print("Copia para ETL creada correctamente")

Copia para ETL creada correctamente


Normalizacion de valor_labo

In [ ]:
df_etl["valor_labo"] = (
    df_etl["valor_labo"]
    .astype("string")
    .str.strip()
    .str.upper()
)

df_etl.loc[df_etl["valor_labo"] == "2", "valor_labo"] = pd.NA

print(df_etl["valor_labo"].value_counts(dropna=False))

valor_labo
LEV     2953
MOD     1399
<NA>     400
SEV       39
Name: count, dtype: Int64


conversion de las fechas al formato adecuado

In [ ]:
columnas_dosaje = [
    "fecha_dosaje_1",
    "fecha_dosaje_2",
    "fecha_dosaje_3",
    "fecha_dosaje_4",
    "fecha_dosaje_5"
]

for columna in columnas_dosaje:
    df_etl[columna] = pd.to_datetime(
        df_etl[columna],
        format="%d/%m/%Y",
        errors="coerce"
    )

correcion del orden cronológico de las fechas de dosaje

In [ ]:
for i in range(1, 5):
    anterior = f"fecha_dosaje_{i}"
    actual = f"fecha_dosaje_{i+1}"

    condicion = (
        df_etl[anterior].notna() &
        df_etl[actual].notna() &
        (df_etl[actual] < df_etl[anterior])
    )

    df_etl.loc[condicion, actual] = pd.NaT

Corrección de fechas de suplementación

In [ ]:
columnas_sup = [
    "fecha_supT1",
    "fecha_supT2",
    "fecha_supT3"
]

for columna in columnas_sup:
    df_etl[columna] = pd.to_datetime(
        df_etl[columna],
        format="%d/%m/%Y",
        errors="coerce"
    )

for i in range(1, 3):
    anterior = f"fecha_supT{i}"
    actual = f"fecha_supT{i+1}"

    condicion = (
        df_etl[anterior].notna() &
        df_etl[actual].notna() &
        (df_etl[actual] < df_etl[anterior])
    )

    df_etl.loc[condicion, actual] = pd.NaT

Correccion de los valores de hemoglobina atipicos

In [ ]:
columnas_hb = [
    "hemoglobina_1",
    "hemoglobina_2",
    "hemoglobina_3",
    "hemoglobina_4",
    "hemoglobina_5"
]

for columna in columnas_hb:
    df_etl[columna] = pd.to_numeric(
        df_etl[columna],
        errors="coerce"
    )

    df_etl.loc[
        (df_etl[columna] < 0) | (df_etl[columna] > 20),
        columna
    ] = pd.NA

Enriquecimiento para las fechas de dosaje

In [ ]:
df_etl["Anio"] = df_etl["fecha_dosaje_1"].dt.year
df_etl["Mes"] = df_etl["fecha_dosaje_1"].dt.month
df_etl["Trimestre"] = df_etl["fecha_dosaje_1"].dt.quarter

df_etl[
    ["fecha_dosaje_1", "Anio", "Mes", "Trimestre"]
].head()

,fecha_dosaje_1,Anio,Mes,Trimestre
0,2024-12-30,2024.0,12.0,4.0
1,2024-03-27,2024.0,3.0,1.0
2,2025-02-14,2025.0,2.0,1.0
3,2025-03-08,2025.0,3.0,1.0
4,2023-11-17,2023.0,11.0,4.0


Preparacion para la conexión con SQL Server

Python prepara los datos transformados y genera archivos CSV para cargar en SQL Server.

Crear un CSV para Dim_Tiempo

In [ ]:
dim_tiempo = (
    df_etl[["fecha_dosaje_1", "Anio", "Mes", "Trimestre"]]
    .dropna(subset=["fecha_dosaje_1"])
    .drop_duplicates()
    .sort_values("fecha_dosaje_1")
)

dim_tiempo["Id_Tiempo"] = range(1, len(dim_tiempo) + 1)

dim_tiempo = dim_tiempo[
    ["Id_Tiempo", "fecha_dosaje_1", "Anio", "Mes", "Trimestre"]
]

dim_tiempo.head()

,Id_Tiempo,fecha_dosaje_1,Anio,Mes,Trimestre
348,1,2023-10-09,2023.0,10.0,4.0
8,2,2023-10-10,2023.0,10.0,4.0
391,3,2023-10-11,2023.0,10.0,4.0
1389,4,2023-10-12,2023.0,10.0,4.0
339,5,2023-10-13,2023.0,10.0,4.0


Crear Dim_Financiador

In [ ]:
dim_financiador = (
    df_etl[["Id_Financiador"]]
    .dropna()
    .drop_duplicates()
    .sort_values("Id_Financiador")
)

dim_financiador.head()

,Id_Financiador
7,1
0,2
866,3
1603,8
2205,9


Crear Dim_ClasificacionAnemia

In [ ]:
dim_clasificacion = pd.DataFrame({
    "Id_Clasificacion": [1, 2, 3],
    "Codigo_Clasificacion": ["LEV", "MOD", "SEV"],
    "Descripcion": [
        "Leve",
        "Moderada",
        "Severa"
    ]
})

dim_clasificacion

,Id_Clasificacion,Codigo_Clasificacion,Descripcion
0,1,LEV,Leve
1,2,MOD,Moderada
2,3,SEV,Severa


Preparacion para la tabla de hechos

Relacionar Id_Tiempo

In [ ]:
fact_gestantes = df_etl.copy()

fact_gestantes = fact_gestantes.merge(
    dim_tiempo[["Id_Tiempo", "fecha_dosaje_1"]],
    on="fecha_dosaje_1",
    how="left"
)

print("Registros:", len(fact_gestantes))

Registros: 4791


Relacionar Id_Clasificacion

In [ ]:
fact_gestantes = fact_gestantes.merge(
    dim_clasificacion[
        ["Id_Clasificacion", "Codigo_Clasificacion"]
    ],
    left_on="valor_labo",
    right_on="Codigo_Clasificacion",
    how="left"
)

Seleccion las columnas de la tabla de hechos

In [ ]:
fact_gestantes = fact_gestantes[
    [
        "Id_Tiempo",
        "Id_Financiador",
        "Id_Clasificacion",
        "cod_pc",
        "hemoglobina_1",
        "hemoglobina_2",
        "hemoglobina_3",
        "hemoglobina_4",
        "hemoglobina_5"
    ]
]

fact_gestantes["CantidadRegistros"] = 1

fact_gestantes.head()

,Id_Tiempo,Id_Financiador,Id_Clasificacion,cod_pc,hemoglobina_1,hemoglobina_2,hemoglobina_3,hemoglobina_4,hemoglobina_5,CantidadRegistros
0,419.0,2,1.0,*****728,12.5,12.6,NaN,NaN,NaN,1
1,160.0,2,2.0,*****636,9.7,NaN,NaN,NaN,NaN,1
2,460.0,2,1.0,*****786,12.8,10.6,13.5,NaN,NaN,1
3,481.0,2,1.0,*****917,12.3,12.3,12.4,NaN,NaN,1
4,38.0,2,1.0,*****096,12.8,12.8,12.8,13.9,NaN,1


Comprobar el resultado

In [ ]:
print("Registros:", len(fact_gestantes))
print("Columnas:", len(fact_gestantes.columns))

fact_gestantes.info()

Registros: 4791
Columnas: 10
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4791 entries, 0 to 4790
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Id_Tiempo          2698 non-null   float64
 1   Id_Financiador     4791 non-null   int64  
 2   Id_Clasificacion   4391 non-null   float64
 3   cod_pc             4791 non-null   object 
 4   hemoglobina_1      2302 non-null   float64
 5   hemoglobina_2      1463 non-null   float64
 6   hemoglobina_3      978 non-null    float64
 7   hemoglobina_4      811 non-null    float64
 8   hemoglobina_5      275 non-null    float64
 9   CantidadRegistros  4791 non-null   int64  
dtypes: float64(7), int64(2), object(1)
memory usage: 374.4+ KB


**Generación de archivos para la carga**

Preparar Dim_Tiempo

In [ ]:
dim_tiempo_carga = dim_tiempo.rename(
    columns={
        "fecha_dosaje_1": "Fecha"
    }
)

dim_tiempo_carga = dim_tiempo_carga[
    [
        "Id_Tiempo",
        "Fecha",
        "Anio",
        "Mes",
        "Trimestre"
    ]
]

dim_tiempo_carga.head()

,Id_Tiempo,Fecha,Anio,Mes,Trimestre
348,1,2023-10-09,2023.0,10.0,4.0
8,2,2023-10-10,2023.0,10.0,4.0
391,3,2023-10-11,2023.0,10.0,4.0
1389,4,2023-10-12,2023.0,10.0,4.0
339,5,2023-10-13,2023.0,10.0,4.0


Preparar Dim_Financiador

In [ ]:
dim_financiador_carga = dim_financiador[
    ["Id_Financiador"]
].copy()

dim_financiador_carga.head()

,Id_Financiador
7,1
0,2
866,3
1603,8
2205,9


Preparar Dim_ClasificacionAnemia

In [ ]:
dim_clasificacion_carga = dim_clasificacion[
    [
        "Id_Clasificacion",
        "Codigo_Clasificacion",
        "Descripcion"
    ]
].copy()

dim_clasificacion_carga

,Id_Clasificacion,Codigo_Clasificacion,Descripcion
0,1,LEV,Leve
1,2,MOD,Moderada
2,3,SEV,Severa


Preparar Fact_Gestantes

In [ ]:
fact_carga = fact_gestantes.copy()

fact_carga.head()

,Id_Tiempo,Id_Financiador,Id_Clasificacion,cod_pc,hemoglobina_1,hemoglobina_2,hemoglobina_3,hemoglobina_4,hemoglobina_5,CantidadRegistros
0,419.0,2,1.0,*****728,12.5,12.6,NaN,NaN,NaN,1
1,160.0,2,2.0,*****636,9.7,NaN,NaN,NaN,NaN,1
2,460.0,2,1.0,*****786,12.8,10.6,13.5,NaN,NaN,1
3,481.0,2,1.0,*****917,12.3,12.3,12.4,NaN,NaN,1
4,38.0,2,1.0,*****096,12.8,12.8,12.8,13.9,NaN,1


Exportación de los 4 archivos

In [ ]:
dim_tiempo_carga.to_csv(
    "Dim_Tiempo.csv",
    index=False,
    sep=";"
)

dim_financiador_carga.to_csv(
    "Dim_Financiador.csv",
    index=False,
    sep=";"
)

dim_clasificacion_carga.to_csv(
    "Dim_ClasificacionAnemia.csv",
    index=False,
    sep=";"
)

fact_carga.to_csv(
    "Fact_Gestantes.csv",
    index=False,
    sep=";"
)

print("Los 4 archivos CSV fueron generados correctamente.")

Los 4 archivos CSV fueron generados correctamente.


Descarga de los archivos

In [ ]:
from google.colab import files

for archivo in [
    "Dim_Tiempo.csv",
    "Dim_Financiador.csv",
    "Dim_ClasificacionAnemia.csv",
    "Fact_Gestantes.csv"
]:
    files.download(archivo)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Corrección de dim tiempo

In [ ]:
dim_tiempo_carga["Anio"] = dim_tiempo_carga["Anio"].astype("Int64")
dim_tiempo_carga["Mes"] = dim_tiempo_carga["Mes"].astype("Int64")
dim_tiempo_carga["Trimestre"] = dim_tiempo_carga["Trimestre"].astype("Int64")

In [ ]:
dim_tiempo_carga.to_csv(
    "Dim_Tiempo.csv",
    index=False,
    sep=";"
)

In [ ]:
print(dim_tiempo_carga.head(10).to_string(index=False))

 Id_Tiempo      Fecha  Anio  Mes  Trimestre
         1 2023-10-09  2023   10          4
         2 2023-10-10  2023   10          4
         3 2023-10-11  2023   10          4
         4 2023-10-12  2023   10          4
         5 2023-10-13  2023   10          4
         6 2023-10-15  2023   10          4
         7 2023-10-16  2023   10          4
         8 2023-10-17  2023   10          4
         9 2023-10-18  2023   10          4
        10 2023-10-19  2023   10          4


In [ ]:
from google.colab import files

files.download("Dim_Tiempo.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Correcion fact_Gestantes

In [ ]:
fact_carga = df_etl[
    [
        "fecha_dosaje_1",
        "Id_Financiador",
        "valor_labo",
        "cod_pc",
        "hemoglobina_1",
        "hemoglobina_2",
        "hemoglobina_3",
        "hemoglobina_4",
        "hemoglobina_5"
    ]
].copy()

fact_carga.to_csv(
    "Fact_Gestantes.csv",
    index=False,
    sep=";"
)

print(fact_carga.head())
print("Registros:", len(fact_carga))

  fecha_dosaje_1  Id_Financiador valor_labo    cod_pc  hemoglobina_1  \
0     2024-12-30               2        LEV  *****728           12.5   
1     2024-03-27               2        MOD  *****636            9.7   
2     2025-02-14               2        LEV  *****786           12.8   
3     2025-03-08               2        LEV  *****917           12.3   
4     2023-11-17               2        LEV  *****096           12.8   

   hemoglobina_2  hemoglobina_3  hemoglobina_4  hemoglobina_5  
0           12.6            NaN            NaN            NaN  
1            NaN            NaN            NaN            NaN  
2           10.6           13.5            NaN            NaN  
3           12.3           12.4            NaN            NaN  
4           12.8           12.8           13.9            NaN  
Registros: 4791


In [ ]:
from google.colab import files
files.download("Fact_Gestantes.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>